In [ ]:
import torch
import torch.nn as nn
import math

from src.positional_encoding import PositionalEncoding
from src.encoder import Encoder
from src.decoder import Decoder


In [14]:
class Transformer(nn.Module):
    def __init__(self, 
                 src_vocab_size, 
                 tgt_vocab_size, 
                 d_model=512,
                 num_heads=8,
                 d_ff=2048,
                 num_layers=6,
                 max_len=1000, 
                 dropout=0.1):
        super().__init__()
        self.d_model = d_model
        
        #埋め込み層
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model)

        #位置エンコーディング
        self.pos_encoding = PositionalEncoding(d_model, max_len, dropout)

        #エンコーダー、デコーダー
        self.encoder = Encoder(d_model, num_heads, d_ff, num_layers, dropout)
        self.decoder = Decoder(d_model, num_heads, d_ff, num_layers, dropout)

        # 最終出力層
        self.generator = nn.Linear(d_model, tgt_vocab_size)

        # パラメータの初期化
        self._init_parameters()

    def _init_parameters(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def make_src_mask(self, src, pad_idx=0): #パディングマスク
        # src: (batch, src_len)
        src_mask = (src != pad_idx).unsqueeze(1).unsqueeze(2)
        return src_mask  # (batch, 1, 1, src_len)

    def make_tgt_mask(self, tgt, pad_idx=0): #パディングマスク + 因果マスク
        batch_size, tgt_len = tgt.size()
        # パディングマスク
        pad_mask = (tgt != pad_idx).unsqueeze(1).unsqueeze(2)
        # 因果マスク
        causal_mask = torch.tril(
            torch.ones(tgt_len, tgt_len, device=tgt.device)
        ).unsqueeze(0).unsqueeze(0)
        tgt_mask = pad_mask & (causal_mask.bool())
        return tgt_mask  # (batch, 1, tgt_len, tgt_len)

    def forward(self, src, tgt, pad_idx=0):
        # マスク生成
        src_mask = self.make_src_mask(src, pad_idx)
        tgt_mask = self.make_tgt_mask(tgt, pad_idx)

        # エンコーダー側
        src_emb = self.src_embedding(src) * math.sqrt(self.d_model)
        src_emb = self.pos_encoding(src_emb)
        enc_output = self.encoder(src_emb, src_mask)

        # デコーダー側
        tgt_emb = self.tgt_embedding(tgt) * math.sqrt(self.d_model)
        tgt_emb = self.pos_encoding(tgt_emb)
        dec_output = self.decoder(tgt_emb, enc_output, src_mask, tgt_mask)

        # 最終出力
        output = self.generator(dec_output)
        return output  # (batch, tgt_len, tgt_vocab_size)


In [12]:
import torch

# Transformer全体のテスト
model = Transformer(
    src_vocab_size=1000,
    tgt_vocab_size=1000,
    d_model=512,
    num_heads=8,
    d_ff=2048,
    num_layers=6
)

# ダミー入力
src = torch.randint(1, 1000, (2, 20))  # (batch=2, src_len=20)
tgt = torch.randint(1, 1000, (2, 15))  # (batch=2, tgt_len=15)

output = model(src, tgt)
print("入力(src)形状:", src.shape)
print("入力(tgt)形状:", tgt.shape)
print("出力形状:", output.shape)
total_params = sum(p.numel() for p in model.parameters())
print(f"総パラメータ数: {total_params:,}")

入力(src)形状: torch.Size([2, 20])
入力(tgt)形状: torch.Size([2, 15])
出力形状: torch.Size([2, 15, 1000])
総パラメータ数: 45,677,544
